# Lab1 — PyTorch Foundations for Computer Vision

**Course**: Deep Learning for Image Analysis

**Class**: M2 IASD App  

**Professor**: Mehyar MLAWEH

---

## Objectives
By the end of this lab, you should be able to:

- Understand how **neurons and layers** are implemented in PyTorch
- Manipulate **tensors** and reason about shapes
- Use **autograd** to compute gradients
- Implement a **training loop** yourself
- Connect theory (neurons, loss, backprop) to actual code

⚠️ This notebook is **intentionally incomplete**.  
Whenever you see **`# TODO`**, you are expected to write code.


**Deadline:** 🗓️ **Saturday, February 7th (23:59)**

## 🤖 A small (honest) note before you start

Let’s be real for a second.

 I know you **can use LLMs (ChatGPT, Copilot, Claude, etc.)** to help you with this lab.  
And yes, **I use them too**, so don’t worry 😄

👉 **You are allowed to use AI tools.**  
But here’s the deal:

- Don’t just **copy–paste** code you don’t understand  
- Take time to **read, question, and modify** what the model gives you  
- If you can solve a block **by yourself, without AI**, that’s excellent

Remember:

> AI can write code for you, but **only you can understand it** — and understanding is what matters for exams, projects, and real work.

Use these tools **as assistants, not as replacements for thinking**.

---

## 📚 Useful documentation (highly recommended)

You will often find answers faster (and more reliably) by checking the official documentation:

- **PyTorch main documentation**  
  https://pytorch.org/docs/stable/index.html

- **PyTorch tensors**  
  https://pytorch.org/docs/stable/tensors.html

- **Neural network modules (`torch.nn`)**  
  https://pytorch.org/docs/stable/nn.html

- **Loss functions** (`BCEWithLogitsLoss`, CrossEntropy, etc.)  
  https://pytorch.org/docs/stable/nn.html#loss-functions

- **Optimizers** (`SGD`, `Adam`, …)  
  https://pytorch.org/docs/stable/optim.html

If you learn how to **navigate the documentation**, you are already thinking like a real AI engineer 👌

---

## PART I

## 0) Colab setup — GPU check

**Instructions**
1. In Colab: `Runtime → Change runtime type to GPU T4`
2. Select **GPU**
3. Save and restart runtime

Then run the cell below.


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# TODO: set the device correctly (cuda if available, else cpu)
# device = ...

# print("Using device:", device)


PyTorch version: 2.9.0+cpu
CUDA available: False


## 1) Imports and reproducibility


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# TODO: fix the random seed for reproducibility
# torch.manual_seed(...)


## 2) PyTorch tensors and shapes

Tensors are multi-dimensional arrays that support:
- GPU acceleration
- automatic differentiation

Understanding **shapes** is critical in deep learning.


In [3]:
# Examples
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.randn(4, 5)

print("a shape:", a.shape)
print("b shape:", b.shape)


a shape: torch.Size([3])
b shape: torch.Size([4, 5])


### 🔍 Question (answer inside the markdown)
- How many dimensions does tensor `b` have?
2 dimensions
- What does each dimension represent conceptually?
First dimension (size 4): the number of rows

Second dimension (size 5): the number of columns

### ✅Tensor operations

Complete the following:

1. Create a tensor `x` of shape `(8, 3)` with random values  
2. Compute:
   - the **mean of each column**
   - the **L2 norm of each row**
3. Normalize `x` **row-wise** using the L2 norm

In [4]:

# 1. Tensor of shape (8, 3)
x = torch.randn(8, 3)

# 2.a Mean of each column
col_mean = x.mean(dim=0)

# 2.b L2 norm of each row
row_l2 = torch.norm(x, p=2, dim=1)

# 3. Row-wise L2 normalization
x_normalized = x / row_l2.unsqueeze(1)

print("x:\n", x)
print("Column mean:", col_mean)
print("Row L2 norm:", row_l2)
print("Row-normalized x:\n", x_normalized)



x:
 tensor([[-1.2383,  0.3860, -0.0995],
        [-1.4218,  0.9596,  1.7097],
        [ 1.4339, -1.0444,  0.8734],
        [-0.2379,  0.2762, -0.9546],
        [ 0.2007,  0.3927, -0.6359],
        [-0.7780,  0.2122, -0.8676],
        [ 1.2133, -1.1041, -0.4611],
        [ 0.5358,  0.7059,  0.9161]])
Column mean: tensor([-0.0365,  0.0980,  0.0601])
Row L2 norm: tensor([1.3009, 2.4219, 1.9773, 1.0218, 0.7738, 1.1845, 1.7041, 1.2746])
Row-normalized x:
 tensor([[-0.9519,  0.2967, -0.0765],
        [-0.5870,  0.3962,  0.7060],
        [ 0.7252, -0.5282,  0.4417],
        [-0.2328,  0.2703, -0.9342],
        [ 0.2593,  0.5074, -0.8217],
        [-0.6568,  0.1791, -0.7325],
        [ 0.7120, -0.6479, -0.2706],
        [ 0.4203,  0.5538,  0.7187]])


## 3) Artificial neuron — from math to code

A neuron computes:

$$
z = \sum_i w_i x_i + b
$$

Then applies an activation function:

$$
y = g(z)
$$

This section connects directly to the theory seen in class.


In [5]:
x = torch.tensor([1.0, -2.0, 3.0])
w = torch.tensor([0.2, 0.4, -0.1])
b = torch.tensor(0.1)

z = torch.sum(x * w) + b
z


tensor(-0.8000)

### Activation functions

1. Implement **ReLU**
2. Implement **Sigmoid**
3. Apply both to `z` and compare the outputs

Which activation preserves negative values?


In [6]:

# ReLU activation
def relu(z):
    return torch.maximum(z, torch.tensor(0.0))

# Sigmoid activation
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

# Example input
z = torch.tensor([-2.0, -0.5, 0.0, 1.0, 2.0])

y_relu = relu(z)
y_sigmoid = sigmoid(z)

y_relu, y_sigmoid


(tensor([0., 0., 0., 1., 2.]),
 tensor([0.1192, 0.3775, 0.5000, 0.7311, 0.8808]))

## 4) Autograd and gradients

PyTorch uses **automatic differentiation** to compute gradients
using the **chain rule** (backpropagation).


In [7]:
x = torch.tensor([1.0, 2.0, -1.0], requires_grad=True)
w = torch.tensor([0.5, -0.3, 0.8], requires_grad=True)
b = torch.tensor(0.2, requires_grad=True)

z = torch.sum(x * w) + b
loss = (z - 1.0) ** 2

loss.backward()

print("loss:", loss.item())
print("grad w:", w.grad)
print("grad b:", b.grad)


loss: 2.890000104904175
grad w: tensor([-3.4000, -6.8000,  3.4000])
grad b: tensor(-3.4000)


### 🔍 Conceptual question

- If `b.grad > 0`, should `b` increase or decrease after a gradient descent step?
Explain **why** in one sentence.
If b.grad > 0, b should decrease after a gradient descent step.Gradient descent updates parameters in the opposite direction of the gradient to reduce the loss, so a positive gradient means decreasing b


## 5) Toy classification dataset

We create a **linearly separable** dataset.

Label rule:
- class = 1 if `x₁ + x₂ + x₃ > 0`
- else class = 0

This mimics a very simple classification problem.


In [8]:

# Parameters
N = 500

# Generate features
X = torch.randn(N, 3)

# Generate labels according to the rule
# class = 1 if x1 + x2 + x3 > 0 else 0
y = (X.sum(dim=1) > 0).float().unsqueeze(1)  # shape (N, 1)

# Train / validation split (80% / 20%)
n_train = int(0.8 * N)

X_train = X[:n_train]
y_train = y[:n_train]

X_val = X[n_train:]
y_val = y[n_train:]

X_train.shape, y_train.shape, X_val.shape, y_val.shape



(torch.Size([400, 3]),
 torch.Size([400, 1]),
 torch.Size([100, 3]),
 torch.Size([100, 1]))

## 6) Model definition

We define a small **MLP** (fully-connected network):

`3 → 16 → 8 → 1`

Activation: ReLU  
Output: raw logits (no sigmoid)


In [9]:

import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 16),   # 3 → 16
            nn.ReLU(),
            nn.Linear(16, 8),   # 16 → 8
            nn.ReLU(),
            nn.Linear(8, 1)     # 8 → 1 (logits)
        )

    def forward(self, x):
        return self.net(x)

# Create model and move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)

model


MLP(
  (net): Sequential(
    (0): Linear(in_features=3, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)

###  parameters

1. Compute **by hand** the total number of parameters
2. Verify your answer using PyTorch

Réseau : 3 → 16 → 8 → 1

Couche 1 (3 → 16)

Poids :
3
×
16
=
48
3×16=48

Biais :
16
16
Total = 64

Couche 2 (16 → 8)

Poids :
16
×
8
=
128
16×8=128

Biais :
8
8
Total = 136

Couche 3 (8 → 1)

Poids :
8
×
1
=
8
8×1=8

Biais :
1
1
Total = 9

 Nombre total de paramètres :

64
+
136
+
9
=
209
64+136+9=
209
	​



In [10]:
total_params = sum(p.numel() for p in model.parameters())
total_params


209

## 7) Training loop

You must complete the full training loop:
- forward pass
- loss computation
- backward pass
- optimizer step

Loss: `BCEWithLogitsLoss`
Optimizer: `SGD`


In [11]:
import torch.optim as optim

# Move data to device
X_train_d = X_train.to(device)
y_train_d = y_train.to(device)
X_val_d   = X_val.to(device)
y_val_d   = y_val.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(20):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    logits = model(X_train_d)

    # Loss computation
    loss = criterion(logits, y_train_d)

    # Backward pass
    loss.backward()

    # Parameter update
    optimizer.step()

    if epoch % 5 == 0:
        print("Epoch", epoch, "| loss =", float(loss))


Epoch 0 | loss = 0.7013431787490845
Epoch 5 | loss = 0.6799149513244629
Epoch 10 | loss = 0.6586374044418335
Epoch 15 | loss = 0.6345996260643005


/tmp/ipython-input-3259597647.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Epoch", epoch, "| loss =", float(loss))


## 8) Evaluation

1. Apply `sigmoid` to the logits
2. Convert probabilities to predictions
3. Compute **accuracy** on the validation set


In [12]:
model.eval()

with torch.no_grad():
    # Forward pass on validation set
    logits = model(X_val_d)

    # Sigmoid to get probabilities
    probs = torch.sigmoid(logits)

    # Convert probabilities to binary predictions
    preds = (probs >= 0.5).float()

# Compute accuracy
accuracy = (preds == y_val_d).float().mean()

accuracy



tensor(0.6900)

### 1. Why do we **not** apply sigmoid inside the model?

Because `BCEWithLogitsLoss` already includes a sigmoid internally, which is **more numerically stable** than applying sigmoid separately.

---

### 2. What would happen if we removed all ReLU activations?

The network would reduce to a **single linear model**, so multiple layers would collapse into one and the model would lose its ability to learn non-linear decision boundaries.

---

### 3. How does this toy problem relate to image classification?

Each input feature plays the role of a **pixel or learned feature**, and the MLP mimics how neural networks combine many features to make a final classification decision.


## 10) Bridge to Computer Vision

So far:
- inputs = vectors of size 3
- layers = fully-connected

Next session:
- inputs = images `(B, C, H, W)`
- layers = convolutions
- same training logic

👉 **Architecture changes, learning principles stay the same.**


## Part II — Training on MNIST

Check the next notebook